In [3]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

TERRITORY = "France"   # SalesTerritory.Name
CATEGORY = "Bikes"     # ProductCategory.Name

engine = create_engine(
f"mysql+mysqlconnector://sqladmin:{os.environ['AWC_DB_PW']}"
"@localhost/Adventureworks2019?charset=utf8mb4")

# One row = one order line. Detail sets the grain (the many side).

# Lookups are LEFT JOINs: product -> subcategory -> category may be empty.

BASE = """
SELECT d.SalesOrderID, d.SalesOrderDetailID, d.ProductID,
d.OrderQty, d.UnitPrice, d.UnitPriceDiscount, d.LineTotal,
h.OrderDate, h.OnlineOrderFlag, h.CustomerID, h.TerritoryID,
h.SalesPersonID, h.SubTotal, h.TaxAmt, h.Freight, h.TotalDue,
p.Name AS ProductName, p.StandardCost, p.ListPrice,
p.ProductSubcategoryID,
sc.Name AS SubCategory,
c.Name  AS Category,
t.Name  AS TerritoryName, t.CountryRegionCode, t.Group AS TerritoryGroup
FROM Sales_SalesOrderDetail d
JOIN Sales_SalesOrderHeader h ON h.SalesOrderID = d.SalesOrderID
JOIN Production_Product p     ON p.ProductID = d.ProductID
LEFT JOIN Production_ProductSubcategory sc
ON sc.ProductSubcategoryID = p.ProductSubcategoryID
LEFT JOIN Production_ProductCategory c
ON c.ProductCategoryID = sc.ProductCategoryID
LEFT JOIN Sales_SalesTerritory t ON t.TerritoryID = h.TerritoryID
"""

ORDER = " ORDER BY d.SalesOrderID, d.SalesOrderDetailID"

# Territory filter only (before the category filter), needed for Task 1c.

terr = pd.read_sql(
text(BASE + " WHERE t.Name = :terr" + ORDER), engine,
params={"terr": TERRITORY})

# The slice: territory x category.

flat = pd.read_sql(
text(BASE + " WHERE t.Name = :terr AND c.Name = :cat" + ORDER), engine,
params={"terr": TERRITORY, "cat": CATEGORY})
flat.to_csv("extract_raw.csv", index=False)

# --- Task 1b: counts that go into report and manifest ----

dates = pd.to_datetime(flat["OrderDate"])
print("lines     :", len(flat))
print("orders    :", flat["SalesOrderID"].nunique())
print("customers :", flat["CustomerID"].nunique())
print("lines_online  :", int((flat["OnlineOrderFlag"] == 1).sum()))
print("lines_reseller:", int((flat["OnlineOrderFlag"] == 0).sum()))
print("first / last OrderDate:", dates.min(), "/", dates.max())
print("months_covered:", dates.dt.to_period("M").nunique())

# --- Task 1c: which join loses rows silently? ----

# Product -> Subcategory: an INNER JOIN would drop lines whose product has no

# subcategory. Counted on the territory (before the category filter).

no_subcat = int(terr["ProductSubcategoryID"].isna().sum())
print(f"1c: lines in {TERRITORY} without subcategory (INNER would lose):",
no_subcat, "of", len(terr))

lines     : 2618
orders    : 1600
customers : 847
lines_online  : 1444
lines_reseller: 1174
first / last OrderDate: 2011-05-30 20:30:00 / 2014-05-29 20:30:00
months_covered: 37
1c: lines in France without subcategory (INNER would lose): 0 of 9088


In [2]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

def hidden_mask(s):
    """True, wo ein Wert wie ein Fehlwert aussieht, aber nicht als NaN zählt."""
    m = pd.Series(False, index=s.index)
    if pd.api.types.is_datetime64_any_dtype(s):
        m |= s.dt.normalize().eq(pd.Timestamp("1900-01-01"))
    elif pd.api.types.is_numeric_dtype(s):
        m |= s.eq(-1)
    else:
        t = s.astype(str).str.strip().str.upper()
        m |= s.notna() & t.isin(["NULL", "N/A", "NA", "NONE", ""])
    return m

rows = []
for col in flat.columns:
    s = flat[col]
    vc = s.value_counts(dropna=True)
    try:
        mn, mx = s.min(), s.max()
    except TypeError:
        mn = mx = None
    plain = int(s.isna().sum())
    hidden = int(hidden_mask(s).sum())
    rows.append({
        "column": col, "dtype": str(s.dtype),
        "pct_missing": round(100 * plain / len(s), 2),
        "n_missing_plain": plain,
        "n_hidden": hidden,
        "n_missing_real": plain + hidden,
        "distinct": int(s.nunique()),
        "min": mn, "max": mx,
        "most_frequent": vc.index[0] if len(vc) else None,
    })

profile = pd.DataFrame(rows)
profile.to_csv("profile.csv", index=False)

# Tabelle für den Report (b): nur Spalten mit Befund

befund = profile[(profile.n_missing_plain > 0) | (profile.n_hidden > 0)]
print(befund[["column", "n_missing_plain", "n_hidden", "n_missing_real"]].to_string(index=False))

       column  n_missing_plain  n_hidden  n_missing_real
SalesPersonID             1444         0            1444


In [7]:
n = len(flat)
print("SalesOrderDetailID:", flat["SalesOrderDetailID"].nunique(), "von", n)
print("SalesOrderID      :", flat["SalesOrderID"].nunique(), "von", n)

SalesOrderDetailID: 2618 von 2618
SalesOrderID      : 1600 von 2618


In [8]:
print(flat.groupby("OnlineOrderFlag")["SalesPersonID"].apply(lambda s: s.isna().mean()))
print("lines_online:", int((flat["OnlineOrderFlag"] == 1).sum()))

OnlineOrderFlag
0    0.0
1    1.0
Name: SalesPersonID, dtype: float64
lines_online: 1444


In [9]:
med = flat["LineTotal"].median()
mean = flat["LineTotal"].mean()
print("median:", med, "| mean:", mean, "| mean/median:", mean / med)

median: 2071.4196 | mean: 2335.462703610008 | mean/median: 1.1274696365767745


In [10]:
def iqr_table(s, label):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    below, above = s < lo, s > hi
    return {
        "grain": label, "n": len(s),
        "lower_fence": lo, "upper_fence": hi,
        "below": int(below.sum()), "above": int(above.sum()),
        "outliers_total": int((below | above).sum()),
        "revenue_share_%": round(100 * s[below | above].sum() / s.sum(), 2),
    }

lines = flat["LineTotal"]
orders = flat.groupby("SalesOrderID")["LineTotal"].sum()   # Bestellwert

result = pd.DataFrame([iqr_table(lines, "lines (LineTotal)"),
                       iqr_table(orders, "orders (Summe LineTotal)")])
print(result.to_string(index=False))

                   grain    n  lower_fence  upper_fence  below  above  outliers_total  revenue_share_%
       lines (LineTotal) 2618   -1339.4067    4737.9125      0    204             204            27.53
orders (Summe LineTotal) 1600   -1618.6300    4785.6900      0    114             114            56.97


In [11]:
top3 = orders.sort_values(ascending=False).head(3)
print(top3)

for oid in top3.index:
    sub = flat[flat["SalesOrderID"] == oid]
    print(oid, "| Positionen:", len(sub), "| Menge:", sub["OrderQty"].sum(),
          "| Kanal:", sub["OnlineOrderFlag"].iloc[0], "| Kunde:", sub["CustomerID"].iloc[0],
          "| Datum:", sub["OrderDate"].iloc[0])

SalesOrderID
46981    117880.152401
57150    114404.310000
63284    105928.812000
Name: LineTotal, dtype: float64
46981 | Positionen: 22 | Menge: 165 | Kanal: 0 | Kunde: 30103 | Datum: 2012-06-29 20:30:00
57150 | Positionen: 22 | Menge: 131 | Kanal: 0 | Kunde: 29923 | Datum: 2013-09-29 20:30:00
63284 | Positionen: 22 | Menge: 124 | Kanal: 0 | Kunde: 29923 | Datum: 2013-12-30 19:30:00


In [7]:
q = flat["OrderQty"]; p = flat["UnitPrice"]; d = flat["UnitPriceDiscount"]
print("Menge <= 0           :", int((q <= 0).sum()))
print("Preis <= 0           :", int((p <= 0).sum()))
print("Preis < Standardkost.:", int((p < flat["StandardCost"]).sum()))
print("Duplikate (Detail-ID):", int(flat["SalesOrderDetailID"].duplicated().sum()))
print("LineTotal != Qty*Preis*(1-Rabatt):",
      int(((q * p * (1 - d)) - flat["LineTotal"]).abs().gt(0.01).sum()))
print("Datum:", flat["OrderDate"].min(), "bis", flat["OrderDate"].max())

Menge <= 0           : 0
Preis <= 0           : 0
Preis < Standardkost.: 926
Duplikate (Detail-ID): 0
LineTotal != Qty*Preis*(1-Rabatt): 0
Datum: 2011-05-30 20:30:00 bis 2014-05-29 20:30:00
